# 🩺 BluaDiagnostics — Validação Interativa do RAG

Notebook de exploração para validar o pipeline RAG implementado no Dia 2 da Sprint 2.

**Pré-requisitos:**
1. `blua-ingest` executado (popula `data/chroma_db/`)
2. `.venv` ativada

**O que faz aqui:**
1. Carrega o retriever
2. Roda queries do dia-a-dia clínico
3. Visualiza top chunks com score + metadados
4. Testa filtros por kb_id

In [ ]:
# Setup — rode esta célula uma vez
import sys
from pathlib import Path

# Adiciona raiz do projeto ao sys.path (pq notebook fica em notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag.retriever import get_retriever, format_chunks_for_prompt
from src.tools.buscar_conhecimento import buscar_conhecimento_clinico

retriever = get_retriever()
print('✅ Retriever carregado')

## 1. Teste básico — caso clássico de red flag

In [ ]:
chunks = retriever.retrieve(
    'dor no peito que irradia para o braço esquerdo com suor frio',
    top_k=4,
)

for i, c in enumerate(chunks, 1):
    print(f'\n--- Chunk {i} ---')
    print(f'KB: {c.kb_id} | Seção: {c.section}')
    print(f'Score: {c.score:.3f}')
    print(f'Texto: {c.text_snippet}')

## 2. Filtro por KB — só red flags

In [ ]:
chunks = retriever.retrieve(
    'cefaleia súbita intensa',
    top_k=3,
    kb_filter='kb05',
)

print(f'Recuperados {len(chunks)} chunks (filtrados em kb05_red_flags):\n')
for c in chunks:
    print(f'  [{c.kb_id}] {c.section} (score={c.score:.3f})')
    print(f'    {c.text_snippet}\n')

## 3. Como o agente vai consumir — formato de tool result

In [ ]:
import json

# Esta é a chamada que o LLM faria via function calling
tool_result = buscar_conhecimento_clinico(
    query='posso tomar ibuprofeno com losartana',
    top_k=3,
)

print(json.dumps(tool_result, indent=2, ensure_ascii=False)[:2000])

## 4. Como o contexto entra no prompt

In [ ]:
chunks = retriever.retrieve('teleconsulta pediatria', top_k=2)
context_block = format_chunks_for_prompt(chunks)
print(context_block)

## 5. Query custom — teste o que quiser

Use a célula abaixo para explorar livremente. Boas queries para experimentar:
- 'gestante com sangramento'
- 'ideação suicida'
- 'meu filho está com febre alta há 3 dias'
- 'pressão arterial muito alta'
- 'reação alérgica grave'

In [ ]:
MINHA_QUERY = 'sua query aqui'

chunks = retriever.retrieve(MINHA_QUERY, top_k=4)
for c in chunks:
    print(f'[{c.kb_id}] {c.section} → score {c.score:.3f}')
    print(f'  {c.text_snippet}\n')